In [1]:
# replication setup — added by patch_notebook.py
import os, sys
_HERE = os.path.abspath(os.getcwd())
for p in (os.path.abspath(os.path.join(_HERE, "..", "..", "reference")),
          os.path.abspath(os.path.join(_HERE, ".."))):
    if p not in sys.path:
        sys.path.insert(0, p)

# networkx 3.x removed from_numpy_matrix; reference/stats_count.py still calls it.
# Behavior of from_numpy_array on a 2D ndarray matches the old from_numpy_matrix.
import networkx as _nx
if not hasattr(_nx, "from_numpy_matrix"):
    _nx.from_numpy_matrix = _nx.from_numpy_array

# numpy 1.20 deprecated np.int as an alias for builtin int; 1.24 removed it.
# reference/ripser_count.py still calls .astype(np.int). Restore the alias.
import numpy as _np
if not hasattr(_np, "int"):
    _np.int = int


In [2]:
from collections import defaultdict, Counter
import itertools
import re
import subprocess
import pickle
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn import linear_model, preprocessing, utils, datasets
from sklearn.metrics import accuracy_score, matthews_corrcoef
from matplotlib.pyplot import scatter
import random
from tqdm.notebook import tqdm
from stats_count import *

In [3]:
import warnings
warnings.filterwarnings('ignore')

## Parameters

In [4]:
np.random.seed(42)
random.seed(42)

In [5]:
max_examples_to_train = 10**10
max_tokens_amount  = 128 # The number of tokens to which the tokenized text is truncated / padded.
layers_of_interest = [i for i in range(12)]  # Layers for which attention matrices and features on them are 
                                             # calculated. For calculating features on all layers, leave it be
                                             # [i for i in range(12)].

In [6]:
train_subset = "train"
test_subset  = "test" # dev/valid - for hyperparameters tuning;
                      # test - for final testing after tuning hyperparameters on the dev set.
input_dir = "../data/processed/"   # Name of the directory with .csv file
model_path = "bert-base-uncased"
# You can use either standard or fine-tuned BERT. If you want to use fine-tuned BERT to your current task, save the
# model and the tokenizer with the commands tokenizer.save_pretrained(output_dir); 
# bert_classifier.save_pretrained(output_dir) into the same directory and insert the path to it here.

old_f_train_file  = "../outputs/features/" + train_subset + \
                    "_all_heads_12_layers_s_e_v_c_b0b1_lists_array_6_thrs_MAX_LEN_128_bert-base-uncased.npy"
old_f_test_file   = "../outputs/features/" + test_subset + \
                    "_all_heads_12_layers_s_e_v_c_b0b1_lists_array_6_thrs_MAX_LEN_128_bert-base-uncased.npy"
ripser_train_file = "../outputs/features/" + train_subset + \
                    "_all_heads_12_layers_MAX_LEN_128_bert-base-uncased_ripser.npy"
ripser_test_file = "../outputs/features/" + test_subset + \
                    "_all_heads_12_layers_MAX_LEN_128_bert-base-uncased_ripser.npy"
templ_train_file  = "../outputs/features/" + train_subset + \
                    "_all_heads_12_layers_MAX_LEN_128_bert-base-uncased_template.npy"
templ_test_file   = "../outputs/features/" + test_subset + \
                    "_all_heads_12_layers_MAX_LEN_128_bert-base-uncased_template.npy"

In [7]:
solver  = "lbfgs"
is_dual = False

## Loading data and features

In [8]:
try:
    train_data = pd.read_csv(input_dir + train_subset + ".csv")
    test_data = pd.read_csv(input_dir + test_subset + ".csv")
except:
    train_data = pd.read_csv(input_dir + train_subset + ".tsv", delimiter="\t", header=None)
    train_data.columns = ["0", "labels", "2", "sentence"]
    test_data = pd.read_csv(input_dir + test_subset + ".tsv", delimiter="\t")

In [9]:
if "label" in train_data.columns:
    train_data["labels"] = (train_data["label"] == "generated").astype(int)
    
if "label" in test_data.columns:
    test_data["labels"] = (test_data["label"] == "generated").astype(int)

In [10]:
y_test = list(map(int, test_data["labels"]))
test_data

,sentence,labels
0,iojs Project includes\n\nFull Splash Client (b...,1
1,Woman in critical condition after Westerville ...,0
2,"Second sense 80-43, 100-100 FM6. True visual t...",1
3,Contemporary photo of a cannon and embrasure a...,0
4,"Victoria Legrand hauntingly coos ""Hush, don't ...",0
...,...,...
995,Allegri: 'Dybala must keep it simple'\n\nBy Fo...,0
996,\nA total of 46 buildings have been built in t...,1
997,Click here for the classic Skyrim version of t...,0
998,Despite his invocation of socialist principles...,0


Requirements for .csv files:

* .csv file **train_data** must contain the column named **labels**, otherwise LogReg accuracy will not be estimated.

In [11]:
old_features_train = np.load(old_f_train_file, allow_pickle=True)[:,:,:,:max_examples_to_train,:]
old_features_test  = np.load(old_f_test_file, allow_pickle=True)[:,:,:,:max_examples_to_train,:]
old_features_train.shape

(12, 12, 6, 1000, 6)

In [12]:
ripser_train = np.load(ripser_train_file, allow_pickle=True)[:,:,:max_examples_to_train,:]
ripser_test  = np.load(ripser_test_file, allow_pickle=True)[:,:,:max_examples_to_train,:]
ripser_train.shape

(12, 12, 1000, 14)

In [13]:
templ_train = np.load(templ_train_file, allow_pickle=True)[:,:,:,:max_examples_to_train]
templ_test  = np.load(templ_test_file, allow_pickle=True)[:,:,:,:max_examples_to_train]
templ_train.shape

(12, 12, 6, 1000)

In [14]:
train_data = train_data[:max_examples_to_train]

In [15]:
X_train = []
for i in range(len(train_data)):
    features = np.concatenate((old_features_train[:,:,:,i,:].flatten(),
                               ripser_train[:,:,i,:].flatten(),
                               templ_train[:,:,:,i].flatten()))
    X_train.append(features)
y_train = train_data["labels"]

X_test = []
for i in range(len(test_data)):
    features = np.concatenate((old_features_test[:,:,:,i,:].flatten(),
                               ripser_test[:,:,i,:].flatten(),
                               templ_test[:,:,:,i].flatten()))
    X_test.append(features)
y_test = test_data["labels"]

In [16]:
X_train = X_train[:max_examples_to_train]
train_data = train_data[:max_examples_to_train]

In [17]:
try:
    assert(len(train_data) == len(X_train))
    assert(len(test_data) == len(X_test))
except:
    print("ASSERTION ERROR!!!")

## LogReg by the feautres from all heads

In [18]:
def pred_by_Xy(X_train, y_train, X_test, classifier, verbose=False, scale=True):

    if scale:
        scaler  = preprocessing.StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)

    classifier.fit(X_train, y_train)
    
    if verbose:
        print("train matt:", matthews_corrcoef(y_train, classifier.predict(X_train)))
        print("train acc: ", accuracy_score(y_train, classifier.predict(X_train)))
    
    if scale:
        X_test = scaler.transform(X_test)
        
    return classifier.predict(X_test), \
           matthews_corrcoef(y_train, classifier.predict(X_train)), \
           accuracy_score(y_train, classifier.predict(X_train))

In [19]:
classifier = linear_model.LogisticRegression(solver=solver)

# The classifier with concrete hyperparameters values, which you should insert here.
# For grid search of hyperparameters - see below.

## Grid Search of hyperparameters. Use it on the dev/vaild set!

(**Reminder**: Don't tune hyperparameters on the test set, to not overfit hyperparameters. Tune hyperparameters on the dev/valid set, and then use the best ones on the test set.)

In [20]:
C_range = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2]
max_iter_range = [1, 2, 3, 5, 10, 25, 50, 100, 500, 1000, 2000]

print(C_range, max_iter_range)

[0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2] [1, 2, 3, 5, 10, 25, 50, 100, 500, 1000, 2000]


In [21]:
matt_scores = dict()
acc_scores  = dict()
matt_scores_train = dict()
acc_scores_train  = dict()
results     = dict()

for C in tqdm(C_range):
    for max_iter in max_iter_range:
        classifier = linear_model.LogisticRegression(penalty='l2', C=C, max_iter=max_iter, dual=is_dual,
                                                     solver=solver)

        result, train_matt, train_acc = pred_by_Xy(X_train, y_train, X_test, classifier)
        results[(C, max_iter)] = result

        matt_scores_train[(C, max_iter)] = matthews_corrcoef(result, y_test)
        acc_scores_train[(C, max_iter)]  = accuracy_score(result, y_test)

        try:
            matt_scores[(C, max_iter)] = matthews_corrcoef(result, y_test)
            acc_scores[(C, max_iter)]  = accuracy_score(result, y_test)
            #print("test matt: ", matthews_corrcoef(result, y_test))
            #print("test acc:  ", accuracy_score(result, y_test))
        except:
            #print("Not labeled")
            pass

  0%|          | 0/10 [00:00<?, ?it/s]

### Prints the list of hyperparameters and corresponding matthews corcoef / accuracy of LogReg, trained with these parameters

In [22]:
try:
    for C in tqdm(C_range):
        for max_iter in max_iter_range:
            print(C, max_iter, ", matt:", matt_scores[(C, max_iter)])
            print(C, max_iter, ", acc :", acc_scores[(C, max_iter)])
            print()
    print("---")
    print()
    print("The best Acc score:")
    print()
    print(max(acc_scores.values()))
    print()
    print("The best Matthew score:")
    print()
    print(max(matt_scores.values()))

except:
    print("Data is not labeled")

  0%|          | 0/10 [00:00<?, ?it/s]

0.0001 1 , matt: 0.3178015185574795
0.0001 1 , acc : 0.648

0.0001 2 , matt: 0.46619625957132066
0.0001 2 , acc : 0.728

0.0001 3 , matt: 0.4881313633615713
0.0001 3 , acc : 0.741

0.0001 5 , matt: 0.5401922268039268
0.0001 5 , acc : 0.769

0.0001 10 , matt: 0.5952277431812029
0.0001 10 , acc : 0.796

0.0001 25 , matt: 0.5883878158124555
0.0001 25 , acc : 0.793

0.0001 50 , matt: 0.5883878158124555
0.0001 50 , acc : 0.793

0.0001 100 , matt: 0.5883878158124555
0.0001 100 , acc : 0.793

0.0001 500 , matt: 0.5883878158124555
0.0001 500 , acc : 0.793

0.0001 1000 , matt: 0.5883878158124555
0.0001 1000 , acc : 0.793

0.0001 2000 , matt: 0.5883878158124555
0.0001 2000 , acc : 0.793

0.0005 1 , matt: 0.3178015185574795
0.0005 1 , acc : 0.648

0.0005 2 , matt: 0.46435446092764987
0.0005 2 , acc : 0.727

0.0005 3 , matt: 0.49186756309009894
0.0005 3 , acc : 0.743

0.0005 5 , matt: 0.5623850711561849
0.0005 5 , acc : 0.78

0.0005 10 , matt: 0.6243129941154031
0.0005 10 , acc : 0.811

0.0005 25 